### **D — Dependency Inversion Principal (DIP)**

Definition: "High-level modules should **NOT** depend on low-level modules. Both should depend on abstractions."

And:
- Abstractions should not depend on details.
- Details should depend on abstractions.

**What** this really means in **Human words**
- Business logic ≠ infrastructure
- Service ≠ frameworks
- Code depends on **interfaces**, not implementations.

**BAD DESIGN (Violates DIP)**

Example: Order Processing Service

In [2]:
class MySQLDatabase:
    def save(self, data):
        print("Saved to MySQL")

Closed for Modify

Core Business Logic

In [ ]:
class OrderService:
    def __init__(self):
        self.db = MySQLDatabase() # Hard dependency
    
    def place_order(self, order):
        self.db.save(order)

**Why this implimentation violate DIP**
- Can not switch database because of tightly coupled.
- Hard to test
- Business logic tied to infrastructure.

👉 High-level module depends on low-level module

**Good Design (Follow DIP)**

Using Strategy Design Pattern

**Abstractions (Interfaces)**

In [3]:
class Database:
    def save(self, data):
        raise NotImplementedError

**Low-level Service**
- Depends on Abstractions (Interface)
- Open for extensions

In [4]:
class MySQLDatabase(Database):
    def save(self, data):
        print(f"{data} saved to MySQL")

class PostgresqlDatabase(Database):
    def save(self, data):
        print(f"{data} saved to postgresql")

**High-level Service**
- Closed for modify
- Depends on Abstractions

In [5]:
class OrderService:
    def __init__(self, database: Database): 
        self.db = database
    
    def place_order(self, order):
        self.db.save(order)


**Usage**

In [6]:
db = PostgresqlDatabase()
service = OrderService(db)
service.place_order({"order_id": 1})

{'order_id': 1} saved to postgresql


✔ High-level code never changes <br>
✔ High-level service never depends on Low-level service <br>
✔ Infrastructure swapped freely <br>
✔ Both High-level and Low-level service depends on Abstractions/Interfaces.

**DIP + Strategy Pattern**

Example: Discount Engine

In [7]:
class DiscountStrategy:
    def apply(self, amount):
        raise NotImplementedError

Low-level service
- Open for Extension
- Depends on abstraction/interface

In [8]:
class PercentageDiscount(DiscountStrategy):
    def apply(self, amount):
        return amount * 0.9
    
class FlatDiscount(DiscountStrategy):
    def apply(self, amount):
        return amount - 50

High-level service
- Closed for Modification
- Depend on Interface/abstraction

In [9]:
class CheckoutService:
    def __init__(self, strategy: DiscountStrategy):
        self.strategy = strategy
    
    def checkout(self, amount):
        discount_price = self.strategy.apply(amount)
        return discount_price

In [10]:
strategy = FlatDiscount()
service = CheckoutService(strategy)
amount = 120
discount_price = service.checkout(amount)

print(f"You got ${amount - discount_price} discount!")

You got $50 discount!


✔ DIP achieved <br>
✔ Business logic depends on abstraction

**DIP + Adapter Pattern (External Services)**

**Third-party SDK**

In [19]:
class SendGridSDK:
    def send_mail(self, to, msg):
        print(f"SendGrid email sent to {to} with \nmessage: {msg}")

**Abstraction**

In [20]:
class NotificationServiceInterface:
    def send(self, to, msg):
        raise NotImplementedError

**Low-level Service (Adapter)**
- Open for extension
- Depend on Abstraction

In [21]:
class SendGridAdapter(NotificationServiceInterface):
    def __init__(self):
        self.sendgrid = SendGridSDK()

    def send(self, to, msg):
        return self.sendgrid.send_mail(to, msg)

**High-level Service**
- Close to modify
- Depend on Abstraction/Interface

In [22]:
class NotificationService:
    def __init__(self, provider: NotificationServiceInterface):
        self.provider = provider
    
    def notify(self, to, msg):
        self.provider.send(to, msg)

**Usage**

In [23]:
provider = SendGridAdapter()
notification_service = NotificationService(provider)

notification_service.notify("abc@gmail.com", "Your payment is successful!")

SendGrid email sent to abc@gmail.com with 
message: Your payment is successful!


✔ No vendor lock-in <br>
✔ Easy to switch providers

#### **DIP + Facade Pattern**

**Complex subsystems**

In [24]:
class InventoryService:
    def reserve(self):
        print("Inventory reserved")

class PaymentService:
    def pay(self):
        print("Payment done")

**Facade Abstraction**

In [29]:
class OrderProcessorInterface:
    def process(self, order):
        raise NotImplementedError

**Low-level Service (Facade Implementation)**
- Open for extension
- Depend on Abstraction

In [40]:
class OrderWorkflow(OrderProcessorInterface):
    def __init__(self):
        self.inventory = InventoryService()
        self.payment = PaymentService()

    def process(self, order):
        self.inventory.reserve()
        self.payment.pay() 

        print(f"Order {order.get("id")} processed successfully!")

**High-level**
- Close for Modification
- Depend on Abstraction

In [41]:
class OrderProcessorService:
    def __init__(self, workflow: OrderProcessorInterface):
        self.workflow = workflow
    
    def create(self, order):
        self.workflow.process(order)

**Usage**

In [42]:
workflow = OrderWorkflow()
service = OrderProcessorService(workflow)

service.create({"id": 1})

Inventory reserved
Payment done
Order 1 processed successfully!


✔ Controllers depend on abstraction <br>
✔ Workflow is replaceable

**DIP + Observer Pattern**

In [60]:
class NotificationInterface:
    def send(self, msg):
        raise NotImplementedError

class ObserverInterface:
    def subscribe(self, observer):
        raise NotImplementedError

    def notify(self, msg):
        raise NotImplementedError
        

In [62]:
# low-level service
class EmailService(NotificationInterface):
    def send(self, msg):
        print(f"Email: {msg}")

class SMSService(NotificationInterface):
    def send(self, msg):
        print(f"SMS: {msg}")

class ObserverServiceStrategy(ObserverInterface):
    def __init__(self):
        self.observers = []

    def subscribe(self, observers):
        for observer in observers:
            self.observers.append(observer)
    
    def notify(self, msg):
        for observer in self.observers:
            observer.send(msg)


**High-level Service**

In [63]:
class ObserverNotificationService:
    def __init__(self, observer: ObserverInterface):
        self.observer = observer
    
    def send(self, msg):
        self.observer.notify(msg)

In [64]:
emailSender = EmailService()
sms_service = SMSService()

observer = ObserverServiceStrategy()
observer.subscribe([emailSender, sms_service])

notification_service = ObserverNotificationService(observer)
notification_service.send("EID Bubarak!")

Email: EID Bubarak!
SMS: EID Bubarak!


✔ Events decoupled <br>
✔ Easy to extend

**Why DIP is HUGE for testing?**

In [66]:
class Database:
    def save(self, data):
        raise NotImplementedError

In [67]:
class OrderService:
    def __init__(self, database: Database): 
        self.db = database
    
    def place_order(self, order):
        self.db.save(order)

In [69]:
class FakeDatabase(Database):
    def save(self, data):
        print(f"Order {data["id"]} saved to fake DB")

In [70]:
service = OrderService(FakeDatabase())
service.place_order({"id": 1})

Order 1 saved to fake DB


✔ No real DB <br>
✔ Fast tests <br>
✔ CI-friendly

**DIP Smells** <br>
If you see:
- new inside business logic
- Direct imports of SDKs
- Hard-coded infrastructure

👉 DIP violation

**SOLID Summary (Mental Model)** <br>

| Principle | What it protects     |
| --------- | -------------------- |
| SRP       | Change               |
| OCP       | Extension            |
| LSP       | Substitution         |
| ISP       | Interface purity     |
| DIP       | Dependency direction |